## Goal

This notebook shows the standard research workflow for a TimeAnalysis trading model:

1. Build a pure pandas model.
2. Connect it through a thin Freqtrade strategy adapter.
3. Run one Docker/Freqtrade backtest.
4. Load metrics, trades, and visualizations.

## 1. Project setup

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Find the project root by walking up from a start directory.

    :param start: directory used as the search starting point
    :return: nearest parent directory containing ``pyproject.toml``
    """

    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    msg = "Could not find project root with pyproject.toml"
    raise FileNotFoundError(msg)


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
FREQTRADE_DIR = PROJECT_ROOT / "trading" / "freqtrade"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

WindowsPath('C:/WorkPrograms/TimeAnalysis')

## 2. Model contract

A model should stay independent from Freqtrade. It receives an OHLCV dataframe and returns a dataframe with indicator columns plus two boolean signal columns:

- `long_entry_signal`
- `long_exit_signal`

In [2]:
from time_analysis.models.simple_momentum import (
    ENTRY_SIGNAL_COLUMN,
    EXIT_SIGNAL_COLUMN,
    SmaMomentumModel,
)

sample_candles = pd.DataFrame(
    {
        "open": [100, 101, 102, 103, 104, 105, 106, 107],
        "high": [101, 102, 103, 104, 105, 106, 107, 108],
        "low": [99, 100, 101, 102, 103, 104, 105, 106],
        "close": [100, 101, 102, 103, 104, 103, 102, 101],
        "volume": [10, 12, 14, 13, 15, 16, 15, 14],
    }
)

model = SmaMomentumModel(fast_window=2, slow_window=4)
signals = model.predict(sample_candles)
display(signals[["close", ENTRY_SIGNAL_COLUMN, EXIT_SIGNAL_COLUMN]])

,close,long_entry_signal,long_exit_signal
0,100,False,False
1,101,False,False
2,102,False,False
3,103,False,False
4,104,False,False
5,103,False,False
6,102,False,True
7,101,False,False


## 3. Configure one experiment

The strategy adapter lives in `trading/freqtrade/user_data/strategies/time_analysis_sma_strategy.py`. It calls the pure pandas model from `populate_indicators()`, then maps model signals to Freqtrade `enter_long` and `exit_long` columns.

In [3]:
from time_analysis.backtesting import BacktestExperimentConfig

experiment = BacktestExperimentConfig(
    name="sma_baseline_january_2025",
    strategy="TimeAnalysisSmaStrategy",
    timeframe="5m",
    timerange="20250101-20250201",
    freqtrade_dir=FREQTRADE_DIR,
    download_data=True,
)

experiment

BacktestExperimentConfig(name='sma_baseline_january_2025', strategy='TimeAnalysisSmaStrategy', timeframe='5m', timerange='20250101-20250201', freqtrade_dir=WindowsPath('C:/WorkPrograms/TimeAnalysis/trading/freqtrade'), config_path=None, config_example_path=None, results_dir=None, download_data=True, export_trades=True, extra_backtesting_args=())

## 4. Run the pipeline

This cell creates a local ignored `config.json` from `config.example.json` if needed, downloads candles, runs the backtest, and loads the latest report.

In [4]:
from time_analysis.backtesting import load_latest_backtest_report, run_freqtrade_backtest

RUN_BACKTEST = True

if RUN_BACKTEST:
    report = run_freqtrade_backtest(experiment)
else:
    report = load_latest_backtest_report(
        FREQTRADE_DIR / "user_data" / "backtest_results",
        strategy_name=experiment.strategy,
    )

report.source_path

Exception in thread Thread-5 (_readerthread):
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3568.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3568.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3568.0_x64__qbz5n2kfra8p0\Lib\subprocess.py", line 1615, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3568.0_x64__qbz5n2kfra8p0\Lib\encodings\cp1251.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Unic

WindowsPath('C:/WorkPrograms/TimeAnalysis/trading/freqtrade/user_data/backtest_results/backtest-result-2026-06-10_13-28-15.zip')

## 5. Inspect metrics and trades

In [6]:
from time_analysis.backtesting import build_backtest_dashboard

dashboard = build_backtest_dashboard(report)

metrics = pd.Series(dashboard["metrics"], name="value").to_frame()
display(metrics)
display(dashboard["pairs"])
display(dashboard["trades"].head(20))

,value
strategy,TimeAnalysisSmaStrategy
source_file,backtest-result-2026-06-10_13-28-15.zip
backtest_start,2025-01-01 02:10:00
backtest_end,2025-02-01 00:00:00
total_trades,395
wins,88
draws,0
losses,307
winrate,0.222785
profit_total,-0.246828


,pair,trades,profit_mean,profit_mean_pct,profit_total_abs,profit_total,profit_total_pct,duration_avg,wins,draws,...,cagr,expectancy,expectancy_ratio,sortino,sharpe,calmar,sqn,profit_factor,max_drawdown_account,max_drawdown_abs
0,ETH/USDT,198,-0.002136,-0.21,-123.292416,-0.123292,-12.33,1:32:00,46,0,...,-0.798287,-0.622689,-0.362957,-67.292988,-31.366964,-63.164463,-3.4915,0.527201,0.124305,124.305037
1,BTC/USDT,197,-0.002176,-0.22,-123.535128,-0.123535,-12.35,1:42:00,42,0,...,-0.798966,-0.627082,-0.450537,-75.465165,-40.319929,-62.837475,-4.4994,0.427382,0.125198,125.377135
2,TOTAL,395,-0.002156,-0.22,-246.827544,-0.246828,-24.68,1:37:00,88,0,...,-0.968215,-0.624880,-0.402591,-140.652627,-70.020009,-63.577608,-5.5252,0.482008,0.247238,247.237655


,pair,stake_amount,max_stake_amount,amount,open_date,close_date,open_rate,close_rate,fee_open,fee_close,...,max_rate,is_open,enter_tag,leverage,is_short,open_timestamp,close_timestamp,orders,funding_fees,weekday
0,ETH/USDT,329.971198,329.971198,0.098180,2025-01-01 03:05:00+00:00,2025-01-01 04:10:00+00:00,3360.88,3350.88,0.001,0.001,...,3363.93,False,,1.0,False,2025-01-01 03:05:00+00:00,2025-01-01 04:10:00+00:00,"[{'amount': 0.09818, 'safe_price': 3360.88, 'f...",0.0,2
1,BTC/USDT,329.958092,329.958092,0.003511,2025-01-01 03:10:00+00:00,2025-01-01 04:10:00+00:00,93978.38,93734.99,0.001,0.001,...,94005.79,False,,1.0,False,2025-01-01 03:10:00+00:00,2025-01-01 04:10:00+00:00,"[{'amount': 0.003511, 'safe_price': 93978.38, ...",0.0,2
2,ETH/USDT,328.933374,328.933374,0.098430,2025-01-01 06:15:00+00:00,2025-01-01 06:20:00+00:00,3341.80,3342.47,0.001,0.001,...,3345.55,False,,1.0,False,2025-01-01 06:15:00+00:00,2025-01-01 06:20:00+00:00,"[{'amount': 0.09843, 'safe_price': 3341.8, 'ft...",0.0,2
3,BTC/USDT,328.869496,328.869496,0.003506,2025-01-01 05:55:00+00:00,2025-01-01 06:50:00+00:00,93801.91,93675.44,0.001,0.001,...,93801.92,False,,1.0,False,2025-01-01 05:55:00+00:00,2025-01-01 06:50:00+00:00,"[{'amount': 0.003506, 'safe_price': 93801.91, ...",0.0,2
4,BTC/USDT,328.382535,328.382535,0.003503,2025-01-01 07:45:00+00:00,2025-01-01 08:35:00+00:00,93743.23,93460.99,0.001,0.001,...,93779.06,False,,1.0,False,2025-01-01 07:45:00+00:00,2025-01-01 08:35:00+00:00,"[{'amount': 0.003503, 'safe_price': 93743.23, ...",0.0,2
5,ETH/USDT,328.384721,328.384721,0.098090,2025-01-01 08:05:00+00:00,2025-01-01 08:50:00+00:00,3347.79,3333.80,0.001,0.001,...,3349.27,False,,1.0,False,2025-01-01 08:05:00+00:00,2025-01-01 08:50:00+00:00,"[{'amount': 0.09809, 'safe_price': 3347.79, 'f...",0.0,2
6,ETH/USDT,327.167355,327.167355,0.098140,2025-01-01 10:40:00+00:00,2025-01-01 11:40:00+00:00,3333.68,3334.92,0.001,0.001,...,3343.09,False,,1.0,False,2025-01-01 10:40:00+00:00,2025-01-01 11:40:00+00:00,"[{'amount': 0.09814, 'safe_price': 3333.68, 'f...",0.0,2
7,ETH/USDT,326.982742,326.982742,0.097810,2025-01-01 11:45:00+00:00,2025-01-01 13:40:00+00:00,3343.04,3338.18,0.001,0.001,...,3354.09,False,,1.0,False,2025-01-01 11:45:00+00:00,2025-01-01 13:40:00+00:00,"[{'amount': 0.09781, 'safe_price': 3343.04, 'f...",0.0,2
8,ETH/USDT,326.636786,326.636786,0.097630,2025-01-01 14:50:00+00:00,2025-01-01 15:15:00+00:00,3345.66,3336.99,0.001,0.001,...,3350.84,False,,1.0,False,2025-01-01 14:50:00+00:00,2025-01-01 15:15:00+00:00,"[{'amount': 0.09763, 'safe_price': 3345.66, 'f...",0.0,2
9,BTC/USDT,327.100857,327.100857,0.003503,2025-01-01 10:40:00+00:00,2025-01-01 15:55:00+00:00,93377.35,94499.93,0.001,0.001,...,94512.59,False,,1.0,False,2025-01-01 10:40:00+00:00,2025-01-01 15:55:00+00:00,"[{'amount': 0.003503, 'safe_price': 93377.35, ...",0.0,2


## 6. Visualize the backtest

In [7]:
dashboard["figures"]["equity"].show()
dashboard["figures"]["drawdown"].show()
dashboard["figures"]["daily_profit"].show()
dashboard["figures"]["pair_summary"].show()

## 7. Plug in a new model

Use this checklist for every new model:

1. Add a pure model under `src/time_analysis/models/`.
2. Make the model return `long_entry_signal` and `long_exit_signal` columns.
3. Add or update a Freqtrade adapter under `trading/freqtrade/user_data/strategies/`.
4. Set `strategy` in `BacktestExperimentConfig` to the adapter class name.
5. Run the pipeline and inspect the dashboard.

In [8]:
class MyModelTemplate:
    """Template for a pure pandas signal model.

    Attributes:
        startup_candle_count: candles required before stable signals are emitted
    """

    startup_candle_count = 50

    def predict(self, candles: pd.DataFrame) -> pd.DataFrame:
        """Create model features and long-only entry/exit signals.

        :param candles: OHLCV dataframe with at least a ``close`` column
        :return: dataframe copy with model feature and signal columns
        """

        result = candles.copy()
        result["my_feature"] = result["close"].pct_change().fillna(0.0)
        result["long_entry_signal"] = result["my_feature"] > 0.01
        result["long_exit_signal"] = result["my_feature"] < 0.0
        return result